## Log creation

In [ ]:
#Imports
import numpy as np
import torch
import torchaudio
from torch.utils.data import DataLoader
from torchvision import transforms

#Custom modules
import cAudiotools
import cLogger
import cTransforms

#Create Log
log = cLogger.Log("output/logs/", prefix="test")

log.log_property("torch_version", torch.__version__, show=True)
log.log_property("torchaudio_version", torchaudio.__version__, show=True)

if torch.cuda.is_available():
    log.log_property("device", "cuda", show=True)
    log.log_property("GPU_count", torch.cuda.device_count(), show=True)
    log.log_property("GPU_device", torch.cuda.get_device_name(0), show=True)
else:
    log.log_property("device", "cpu", show=True)

## Define data parameters

In [6]:
#Parameters:
loader_params = {
    "dataset_train": r"C:\Datasets\_compiled\msp-podcast-2\labels_train_VAD.csv",
    "dataset_train_dir": r"C:\Datasets\_compiled\msp-podcast-2\Train",
    "dataset_dev": r"C:\Datasets\_compiled\msp-podcast-2\labels_development_VAD.csv",
    "dataset_dev_dir": r"C:\Datasets\_compiled\msp-podcast-2\Development",
    "batch_size": 16,
    "shuffle": True,
    "collate_function": cAudiotools.Batching.spectrogram_dynamic,
    "target_transform": cTransforms.NormalizeMinus(1, 7)
    }

audio_params = {
    "sample_rate": 16000
}

spectrogram_params = {
    "n_fft": 512,
    "hop_length": 160,
    "win_length": 400,
    "window_fn": torch.hamming_window,
    "normalized": True,
    "power": 2
}

mel_params = {
    "n_mels": 64,
    "pad_mode": "constant",
    "mel_scale": "htk",
    "n_mfcc": 20
}


## Define training parameters

In [7]:

spectogram_transform = torchaudio.transforms.Spectrogram(
    n_fft=spectrogram_params["n_fft"],
    hop_length=spectrogram_params["hop_length"],
    win_length=spectrogram_params["win_length"],
    window_fn=spectrogram_params["window_fn"],
    normalized=spectrogram_params["normalized"],
    power=spectrogram_params["power"]
    )

melspectogram_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=audio_params["sample_rate"],
    n_fft=spectrogram_params["n_fft"],
    win_length=spectrogram_params["win_length"],
    hop_length=spectrogram_params["hop_length"],
    window_fn=spectrogram_params["window_fn"],
    power=spectrogram_params["power"],
    n_mels=mel_params["n_mels"],
    normalized=spectrogram_params["normalized"],
    pad_mode=mel_params["pad_mode"],
    mel_scale=mel_params["mel_scale"]
)

mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=audio_params["sample_rate"],
    n_mfcc=mel_params["n_mfcc"],
    melkwargs={
        "n_fft": spectrogram_params["n_fft"],
        "win_length": spectrogram_params["win_length"],
        "hop_length": spectrogram_params["hop_length"],
        "window_fn": spectrogram_params["window_fn"],
        "power": spectrogram_params["power"],
        "n_mels": mel_params["n_mels"],
        "normalized": spectrogram_params["normalized"],
        "pad_mode": mel_params["pad_mode"],
        "mel_scale": mel_params["mel_scale"]
    }
)
loader_params["data_transform"] = mfcc_transform


log.log_properties("Loader", loader_params)
log.log_properties("Audio", audio_params)
log.log_properties("Spectrogram", spectrogram_params)
log.log_properties("Mel Spectrogram", mel_params)

#Train set
msp_vad_train = cAudiotools.AudioDatasetVAD(
    loader_params["dataset_train"], 
    loader_params["dataset_train_dir"],
    transform=loader_params["data_transform"],
    target_transform=loader_params["target_transform"]
    )
train_dataloader = DataLoader(
    msp_vad_train,
    batch_size=loader_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"]
    )

#Development (validation) set
msp_vad_dev = cAudiotools.AudioDatasetVAD(
    loader_params["dataset_dev"],
    loader_params["dataset_dev_dir"],
    transform=spectogram_transform,
    target_transform=loader_params["target_transform"]
    )
dev_dataloader = DataLoader(
    msp_vad_dev,
    batch_size=loader_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"]
    )


## Data sample

In [ ]:
train_features, train_labels = next(iter(train_dataloader))
log.log_message(f"Feature batch shape: {train_features.size()}", show=True)
log.log_message(f"Labels batch shape: {train_labels.size()}", show=True)

sample = train_features[0]
label = train_labels[0]

spectogram_2_db = torchaudio.transforms.AmplitudeToDB(
    stype='power',
    #top_db=80
    )
#audiotools.Plot.spectrogram(spectogram_2_db(sample), ylabel="Frequency Bin", xlabel="Frame", cmap="viridis")
cAudiotools.Plot.mfcc(sample)
print("MFCC: ", sample)
print(f"Label: {label}")
log.save()